# Model Training (K-Fold Bagging)


## 1. Setup and Configuration

### 1.1. Environment Variables

In [ ]:
import os

# Specify GPU to use (e.g., GPU:0, CPU:-1):
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Allow TensorFlow to allocate GPU memory as needed:
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

# Disable all auto-JIT clustering at the process level:
# os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=-1"

# Enable deterministic operations for reproducibility:
# os.environ["TF_DETERMINISTIC_OPS"] = "1"
# os.environ["TF_CUDNN_DETERMINISTIC"] = "1"
# os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# If it fails to determine best cudnn convolution algorithm:
# os.environ["XLA_FLAGS"] = "--xla_gpu_strict_conv_algorithm_picker=false"

# Suppress TensorFlow logging (1: INFO, 2: WARNING, 3: ERROR):
# os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

### 1.2. Imports

In [ ]:
from _imports import * # Centralized imports for the project
from sklearn.model_selection import KFold



## 2. Run Parameters 

In [ ]:
EPOCHS = 50
BATCH_SIZE = 64
N_SPLITS = 10

DATA_SEED = 0
TRAIN_SEED = 0

# Set Python, NumPy, Keras and TensorFlow seeds
set_random_seed(TRAIN_SEED)

# Reproducibility settings for TensorFlow:
# Note: must have same inputs and hardware
# Warning: this affects overall performance
tf.config.experimental.enable_op_determinism()

# Enable or disable XLA compilation
# Note: some layers don't support determinism with XLA
USE_JIT_COMPILE = False


In [ ]:
POLICY = mixed_precision.Policy('float32')
mixed_precision.set_global_policy(POLICY)

BYTES_PER_PARAM = tf.dtypes.as_dtype(POLICY.variable_dtype).size

In [ ]:
# Set to an existing dir to resume training
RUN_DIR = f"runs/{get_caller_stem()}"  # (e.g. "runs/train_1")

## 3. Data Loading and Preprocessing

In [ ]:
(
    s008_coord_input,
    s008_lidar_input,
    s008_y_train,
    s009_coord_input,
    s009_lidar_input,
    s009_y,
) = load_dataset_sparse_labels(s008_path="./data/s008", s009_path="./data/s009")

NUM_CLASSES = 256


## 4. Model Definition

In [ ]:
def build_model(show_summary: bool = True) -> Model:
    # ———————————————————————————————————————————————————————————————————————————— #
    #                              Model Construction                              #
    # ———————————————————————————————————————————————————————————————————————————— #

    #! Lambda has deserialization issues, so providing the output shape is necessary

    initializer = tf.keras.initializers.GlorotUniform(
        seed=TRAIN_SEED,
    )

    # ———————————————————————————————— LiDAR Input ——————————————————————————————— #
    x_lidar_input = layers.Input(shape=(20, 200, 10), name="lidar_input")

    # Inline one-hot encoding of semantic values
    one_hot_lidar = layers.Lambda(
        lambda x: tf.concat(
            [
                # “Is there a BS anywhere in the 10 channels?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, -2), axis=-1, keepdims=True), tf.float32),
                # “Vehicle?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, -1), axis=-1, keepdims=True), tf.float32),
                # “Obstacle?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, 1), axis=-1, keepdims=True), tf.float32),
                # “Free?” → 1 channel (all channels zero)
                tf.cast(tf.reduce_all(tf.equal(x, 0), axis=-1, keepdims=True), tf.float32),
            ],
            axis=-1,
        ),
        #! Lambda has deserialization issues, so providing the output shape is necessary
        output_shape=(20, 200, 4),
        name="lidar_transform_to_one_hot",
    )(x_lidar_input)
    # -> (batch, 20, 200, 4)

    # Flatten the 20×200 grid into a 4000-length sequence with the 4 channels
    x_lidar_flat: layers.Layer = layers.Reshape((20 * 200, 4), name="lidar_flatten_4_channels")(one_hot_lidar)

    # ———————————————————————————————— GPS Input ———————————————————————————————— #
    x_coord_input = layers.Input(shape=(2,), name="coord_input")

    # Turn (batch,2) → (batch,1,2) → tile to (batch,4000,2)'
    x_coord: layers.Layer = layers.Lambda(
        lambda x: tf.tile(tf.expand_dims(x, axis=1), [1, 20 * 200, 1]),
        #! Lambda has deserialization issues, so providing the output shape is necessary
        output_shape=(20 * 200, 2),
        name="coord_tile_flat",
    )(x_coord_input)

    # ————————————————————————————— Combine Branches ————————————————————————————— #
    # Fuse channels:  (batch,4000,4) + (batch,4000,2) → (batch,4000,6)
    combined = layers.Concatenate(axis=-1, name="combine_lidar_coord")([x_lidar_flat, x_coord])

    # ———————————————————————————————————— CNN ——————————————————————————————————— #
    x = layers.Conv1D(
        filters=128,
        kernel_size=9,
        padding="same",
        data_format="channels_last",
        activation=None,
        use_bias=False,
        kernel_initializer=initializer,
        name="conv1d_0",
    )(combined)
    x = layers.BatchNormalization(name="conv1d_0_bn")(x)
    x = layers.Activation("silu", name="conv1d_0_act")(x)
    x = layers.MaxPooling1D(pool_size=4, name="max_pool_0")(x)

    x = layers.Conv1D(
        filters=256,
        kernel_size=4,
        padding="same",
        data_format="channels_last",
        activation=None,
        use_bias=False,
        kernel_initializer=initializer,
        name="conv1d_1",
    )(x)
    x = layers.BatchNormalization(name="conv1d_1_bn")(x)
    x = layers.Activation("tanh", name="conv1d_1_act")(x)
    x = layers.MaxPooling1D(pool_size=2, name="max_pool_1")(x)

    x = layers.GlobalMaxPooling1D(name="global_max_pooling")(x)

    # ———————————————————————————————————— DNN ——————————————————————————————————— #
    x = layers.Dense(
        units=175,
        activation=None,
        kernel_initializer=initializer,
        name="dense_0",
    )(x)
    x = layers.Activation("tanh", name="dense_0_act")(x)
    x = layers.Dropout(rate=0.1, name="dense_0_dropout")(x)

    # —————————————————————————————————— Output —————————————————————————————————— #
    outputs = layers.Dense(
        256,
        activation="softmax",
        name="output",
        kernel_initializer=initializer,
    )(x)

    # —————————————————————————— Set Inputs and Outputs —————————————————————————— #
    model = Model(inputs=(x_lidar_input, x_coord_input), outputs=(outputs,))

    # ———————————————————————————————— Compilation ——————————————————————————————— #
    model.summary() if show_summary else None

    optimizer = optimizers.AdamW(
        learning_rate=0.0028523343462769487,
        weight_decay=1e-4,
    )

    model.compile(
        optimizer=optimizer,
        loss=losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
        jit_compile=USE_JIT_COMPILE,
    )

    return model


## Main

In [ ]:
try:
    # ——————————————————————————————————— Setup —————————————————————————————————— #
    # Create the run directories used to store models, history, logs, and artifacts.
    (
        study_dir,
        args_dir,
        fig_dir,
        backup_dir,
        history_dir,
        scaler_dir,
        model_dir,
        logs_dir,
        tensorboard_dir,
    ) = init_study_dirs(RUN_DIR, study_name="model_training")

    # ————————————————————————————— K-Fold Preparation ————————————————————————————— #
    # K-Fold here means we split the full s008 dataset into N_SPLITS folds.
    # Each fold is used once as validation, while the remaining folds are training.
    if N_SPLITS < 2:
        raise ValueError("N_SPLITS must be at least 2 for k-fold training.")

    # Clamp the number of folds to the dataset size to avoid invalid splits.
    n_samples = len(s008_y_train)
    print(f"Total samples in s008 dataset: {n_samples}")
    n_splits = min(N_SPLITS, n_samples)
    if n_splits < 2:
        raise ValueError("Not enough samples to run k-fold training.")
    if n_splits != N_SPLITS:
        print(f"Adjusting N_SPLITS from {N_SPLITS} to {n_splits} to fit dataset size.")
    print(f"Using KFold with n_splits={n_splits} on full s008 dataset.")

    # OOF (Out-Of-Fold) predictions hold, for each training sample, the prediction
    # made by the fold where that sample was in the validation set.
    oof_preds = np.zeros((len(s008_y_train), NUM_CLASSES), dtype=np.float32)

    # Accumulate predictions on s009 for each fold; later we average them (bagging).
    test_preds_sum = np.zeros((len(s009_y), NUM_CLASSES), dtype=np.float32)

    # Store per-fold metrics for analysis and debugging.
    fold_metrics = []

    # Plain KFold (not stratified) over the full s008 dataset.
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=DATA_SEED)

    # ——————————————————————————————— Train K-Fold ——————————————————————————————— #
    for fold_idx, (train_idx, val_idx) in enumerate(
        kf.split(s008_lidar_input, s008_y_train),
        start=1,
    ):
        # Clear TF graph/state between folds to avoid memory growth.
        clear_session()
        # Vary seed per fold for reproducibility without identical initializations.
        set_random_seed(TRAIN_SEED + fold_idx)

        # Keep fold outputs isolated in their own subdirectories.
        fold_name = f"fold_{fold_idx}"
        fold_backup_dir = os.path.join(backup_dir, fold_name)
        fold_history_dir = os.path.join(history_dir, fold_name)
        fold_model_dir = os.path.join(model_dir, fold_name)
        fold_scaler_dir = os.path.join(scaler_dir, fold_name)
        fold_logs_dir = os.path.join(logs_dir, fold_name)
        fold_tensorboard_dir = os.path.join(tensorboard_dir, fold_name)

        for _dir in (
            fold_backup_dir,
            fold_history_dir,
            fold_model_dir,
            fold_scaler_dir,
            fold_logs_dir,
            fold_tensorboard_dir,
        ):
            os.makedirs(_dir, exist_ok=True)

        # Slice the fold data by indices for training and validation.
        x_lidar_train = s008_lidar_input[train_idx]
        x_lidar_val = s008_lidar_input[val_idx]
        x_coord_train = s008_coord_input[train_idx]
        x_coord_val = s008_coord_input[val_idx]
        y_train = s008_y_train[train_idx]
        y_val = s008_y_train[val_idx]

        # —————————————————————————————— Data Preprocessing ————————————————————————————— #
        # Fit the scaler on the training split only to avoid leakage.
        coord_scaler = StandardScaler()

        coord_scaler.fit(x_coord_train)
        x_coord_train = coord_scaler.transform(x_coord_train)
        x_coord_val = coord_scaler.transform(x_coord_val)
        # Apply the same scaler to s009 for this fold's model.
        s009_coord_fold = coord_scaler.transform(s009_coord_input)

        # Persist the scaler for reproducibility and inference.
        scaler_path = os.path.join(fold_scaler_dir, "coord_scaler.pkl")
        with open(scaler_path, "wb") as scaler_file:
            pickle.dump(coord_scaler, scaler_file)

        # —————————————————————————————— Train the Model ————————————————————————————— #
        # Only show model summary once to avoid clutter.
        model = build_model(show_summary=fold_idx == 1)

        history = model.fit(
            x=[x_lidar_train, x_coord_train],
            y=y_train,
            validation_data=([x_lidar_val, x_coord_val], y_val),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=get_callbacks_model(
                backup_dir=os.path.join(fold_backup_dir, "training"),
                checkpoint_dir=os.path.join(fold_backup_dir, "checkpoints"),
                early_stopping_patience=10,
                reduce_lr_patience=3,
                #! Can cause high memory usage
                # tensorboard_logs=fold_tensorboard_dir,
            ),
            verbose=2,
        )

        # Save the fold model so each fold can be inspected or ensembled later.
        model.save(os.path.join(fold_model_dir, "model.keras"))

        # ———————————————————————————————— Fold Metrics ———————————————————————————————— #
        # Validation metrics for this fold using its held-out subset.
        val_loss, val_acc = model.evaluate(
            [x_lidar_val, x_coord_val], y_val, batch_size=BATCH_SIZE, verbose=0
        )

        # Store OOF predictions for the validation indices of this fold.
        val_preds = model.predict(
            [x_lidar_val, x_coord_val], batch_size=BATCH_SIZE, verbose=0
        )
        oof_preds[val_idx] = val_preds

        # Predict on s009 and accumulate to build a bagged ensemble.
        test_preds = model.predict(
            [s009_lidar_input, s009_coord_fold], batch_size=BATCH_SIZE, verbose=0
        )
        test_preds_sum += test_preds

        # Best epoch is chosen by highest validation accuracy for this fold.
        best_idx = int(np.argmax(history.history["val_accuracy"]))

        # Save a compact snapshot of fold metrics for inspection.
        fold_metrics.append(
            {
                "fold": fold_idx,
                "best_epoch": best_idx + 1,
                "best_train_loss": float(history.history["loss"][best_idx]),
                "best_val_loss": float(history.history["val_loss"][best_idx]),
                "best_train_accuracy": float(history.history["accuracy"][best_idx]),
                "best_val_accuracy": float(history.history["val_accuracy"][best_idx]),
                "val_loss": float(val_loss),
                "val_accuracy": float(val_acc),
            }
        )

        # ——————————————————————————————— Save history ——————————————————————————————— #
        # Persist per-epoch training/validation curves for this fold.
        history_path = os.path.join(fold_history_dir, "history.csv")
        history_data = {
            "epoch": list(range(1, len(history.history["loss"]) + 1)),
            "train_loss": history.history["loss"],
            "val_loss": history.history["val_loss"],
            "train_accuracy": history.history["accuracy"],
            "val_accuracy": history.history["val_accuracy"],
        }

        history_df = pd.DataFrame(history_data)
        history_df.to_csv(history_path, index=False)

        # Force garbage collection between folds to keep memory stable.
        gc.collect()

    # —————————————————————————————— OOF + Bagged Metrics —————————————————————————————— #
    # OOF accuracy: compare argmax of OOF predictions vs ground-truth labels.
    oof_pred_labels = np.argmax(oof_preds, axis=1)
    oof_accuracy = float(np.mean(oof_pred_labels == s008_y_train))

    # OOF loss: mean cross-entropy over OOF probabilities for each training sample.
    oof_loss = float(
        np.mean(
            tf.keras.losses.sparse_categorical_crossentropy(s008_y_train, oof_preds).numpy()
        )
    )

    # Bagged predictions: average probabilities across folds for s009.
    test_preds_avg = test_preds_sum / n_splits
    test_pred_labels = np.argmax(test_preds_avg, axis=1)
    test_accuracy = float(np.mean(test_pred_labels == s009_y))
    test_loss = float(
        np.mean(
            tf.keras.losses.sparse_categorical_crossentropy(s009_y, test_preds_avg).numpy()
        )
    )

    # Save prediction arrays for later analysis or stacking.
    np.save(os.path.join(args_dir, "oof_preds.npy"), oof_preds)
    np.save(os.path.join(args_dir, "test_preds_avg.npy"), test_preds_avg)

    # Save per-fold metrics in a single CSV.
    fold_metrics_df = pd.DataFrame(fold_metrics)
    fold_metrics_df.to_csv(os.path.join(args_dir, "fold_metrics.csv"), index=False)

    # ———————————————————————————————— Model Stats ——————————————————————————————— #
    # Note: model_stats uses the last fold's model, but includes global OOF/bagged metrics.
    write_model_stats_to_file(
        model=model,
        file_path=os.path.join(args_dir, "model_stats.txt"),
        batch_size=BATCH_SIZE,
        bytes_per_param=tf.dtypes.as_dtype(POLICY.variable_dtype).size,
        device="gpu/0",
        stats_to_measure=(
            "parameters",
            "model_size",
            "flops",
            "macs",
            "summary",
            "inference_latency",
            # "cpu_util_percent",
            # "cpu_power_rapl_w",
            # "ram_used_bytes",
            # "ram_util_percent",
            # "gpu_util_percent",
            # "gpu_mem_used_bytes",
            # "gpu_power_w",
        ),
        extra_attrs={
            "n_splits": n_splits,
            "oof_loss": oof_loss,
            "oof_accuracy": oof_accuracy,
            "test_loss_s009_bagged": test_loss,
            "test_accuracy_s009_bagged": test_accuracy,
        },
        test_runs=10,
        verbose=1,
    )
    # ———————————————————————————————————————————————————————————————————————————— #
except Exception as e:
    print(f"An error occurred: {e}")
    traceback.print_exc()

    with open(os.path.join(logs_dir, "training_error.log"), "a") as f:
        f.write(f"An error occurred during training: {e} {traceback.format_exc()}")
finally:
    # Clean up directories
    shutil.rmtree(backup_dir, ignore_errors=True)
    if not os.listdir(logs_dir):
        os.rmdir(logs_dir)